# Jaldi — GPU stages (Colab A100)

Runs the parts of the pipeline that need a GPU and writes everything to Google Drive (`MyDrive/jaldi_data`).

| Session | Stages | Why separate |
|---|---|---|
| **A** | corpus → script check → TTS + pauses → audio partials (Whisper / IndicConformer, turn detector, prosody) | `parler-tts` pins its own `transformers` version |
| **B** (restart runtime first) | LLM labels (vLLM) → time-to-ready → optional text encoder → features → train → simulate → report | `vllm` needs a newer `transformers` |

Every stage ends with a gate; if one fails, stop and note it in `PROGRESS.md`. None of this has been run end to end yet; expect to fix small things on the first run.

## Setup (both sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/jaldi_data'
RES_BACKUP = f'{DATA}/results_backup'
!mkdir -p {DATA} {RES_BACKUP}
!git clone https://github.com/tanuku-saikarthik/jaldi /content/jaldi 2>/dev/null || (cd /content/jaldi && git pull)
%cd /content/jaldi
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Session A — audio

### A1. Install

In [ ]:
!pip -q install -e ".[gpu]"
!pip -q install git+https://github.com/huggingface/parler-tts.git
!pip -q install soundfile scipy

### A2. Stage 1: corpus (gate: ≥1,000 turns, all gold actions valid)

In [ ]:
!python -m jaldi.cli --data-dir {DATA} corpus

### A3. Stage 2a: Hinglish script check
Synthesizes 10 Hinglish turns in romanized vs mixed script and checks which one survives ASR better.
Also check the Indic Parler-TTS speaker names in `jaldi/corpus/lexicon.py::SPEAKERS` against the model card first.

In [ ]:
!python -m jaldi.cli --data-dir {DATA} script-check --n 10
import json; SCRIPT = json.load(open('results/script_check.json'))['pick']; print('Hinglish script:', SCRIPT)

### A4. Stage 2b: benchmark TTS throughput on 20 turns, then size N to ~90 GPU-minutes

In [ ]:
!python -m jaldi.cli --data-dir {DATA} tts --hinglish-script {SCRIPT} --limit 20 --benchmark

In [ ]:
# read segments/second from the output above
SEG_PER_SEC = 1.0          # <- replace with the measured value
SEG_PER_TURN = 3.35        # average chunks per turn in the corpus (results/corpus_stats.json run)
N = min(1800, int(90 * 60 * SEG_PER_SEC / SEG_PER_TURN))
print('turns to synthesize:', N)

In [ ]:
!python -m jaldi.cli --data-dir {DATA} tts --hinglish-script {SCRIPT} --limit {N}

### A5. Stage 3: audio partials (prefix ASR every 200 ms + VAD + prosody + turn detector)
Gate: final partial ≈ full-clip transcript on ≥80% of turns.
Optional comparison for Hindi: run once with `--asr-hi indicconformer --limit 100` into a scratch data dir and compare.

In [ ]:
!python -m jaldi.cli --data-dir {DATA} partials --mode audio --asr-hi whisper

In [ ]:
!cp results/*.json {RES_BACKUP}/ && ls {DATA}

# Session B — labels and model

**Runtime → Restart session**, run the Setup cell again, then:

In [ ]:
!pip -q install -e ".[gpu,labels]"
!cp {RES_BACKUP}/*.json results/ 2>/dev/null; ls results

### B1. Stage 4: LLM action labels (gate: final action == gold on ≥90%) + M1 analysis

In [ ]:
!python -m jaldi.cli --data-dir {DATA} label --extractor llm --llm-model Qwen/Qwen2.5-7B-Instruct

### B2. Measure time-to-ready (replaces the 600 ms assumption)

In [ ]:
!python -m jaldi.cli --data-dir {DATA} measure-ttr --llm-model Qwen/Qwen2.5-7B-Instruct

### B3. Optional: model B text encoder (MuRIL) for fusion

In [ ]:
# !python -m jaldi.cli --data-dir {DATA} encoder --encoder-model google/muril-base-cased --epochs 2

### B4. Stages 5–7 + report (CPU is fine)

In [ ]:
!python -m jaldi.cli --data-dir {DATA} features
!python -m jaldi.cli --data-dir {DATA} train
!python -m jaldi.cli --data-dir {DATA} simulate
!python -m jaldi.cli --data-dir {DATA} report
!cp -r results {DATA}/results_final

### B5. Bring results home
Copy `MyDrive/jaldi_data/results_final/*` into the repo's `results/` and the updated `README.md` results block, then commit:
`stage 9: GPU run results`. The README banner switches from "Smoke run" automatically because `labels_meta.json` now says `mode=audio`.